In [32]:
import pandas as pd
import numpy as np
import string
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout,Bidirectional
from tensorflow.keras.callbacks import EarlyStopping



[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/harshsisodiya678/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [33]:
#load data 
df = pd.read_csv("Spam_SMS.csv")

df.head()
df['lable'] = df['Class'].map({'ham':0 , 'spam':1})
print(df['Class'].value_counts())

Class
ham     4827
spam     747
Name: count, dtype: int64


In [34]:
#preprocess Text 

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()

    #remove punctuation 
    text = text.translate(
        str.maketrans('' , '' , string.punctuation)
    )

    #remove stopwords 
    words = text.split()
    words = [w for w in words if w not in stop_words]

    return ' '.join(words)

df['cleaned'] = df['Message'].apply(clean_text)

print("Original :", df['Message'][2])
print("Cleaned  :", df['cleaned'][2])


Original : Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
Cleaned  : free entry 2 wkly comp win fa cup final tkts 21st may 2005 text fa 87121 receive entry questionstd txt ratetcs apply 08452810075over18s


In [35]:
#encode labels 

df['label'] = df['Class'].map({'ham': 0, 'spam': 1})

print("Label distribution:")
print(df['label'].value_counts())

X = df['cleaned'].values
y = df['label'].values

Label distribution:
label
0    4827
1     747
Name: count, dtype: int64


In [36]:
#tokenize + pad 

VOCAB_SIZE = 10000
MAX_LEN    = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE , oov_token=
"<OOV>")
tokenizer.fit_on_texts(df['cleaned'])

X = pad_sequences(
        tokenizer.texts_to_sequences(df['cleaned']),
        maxlen=MAX_LEN, padding='post'
    )
y = df['label'].values




In [39]:
#split 

from sklearn.utils.class_weight import compute_class_weight
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("\nClass weights:", class_weight_dict)


Class weights: {0: np.float64(0.5756519493932352), 1: np.float64(3.804607508532423)}


In [40]:
model = Sequential([
    Embedding(VOCAB_SIZE , 32 ,input_length=MAX_LEN ),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [41]:
model.compile(
              optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy']
)

In [42]:
print(df.shape)
print(X_train.shape)

(5574, 5)
(4459, 100)


In [43]:
model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weight_dict,
    verbose=1
)

Epoch 1/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 6s 34ms/step - accuracy: 0.8839 - loss: 0.3765 - val_accuracy: 0.9686 - val_loss: 0.1074
Epoch 2/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - accuracy: 0.9529 - loss: 0.2068 - val_accuracy: 0.9619 - val_loss: 0.1144
Epoch 3/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - accuracy: 0.9905 - loss: 0.0613 - val_accuracy: 0.9641 - val_loss: 0.0981
Epoch 4/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - accuracy: 0.9955 - loss: 0.0288 - val_accuracy: 0.9731 - val_loss: 0.0833
Epoch 5/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - accuracy: 0.9963 - loss: 0.0201 - val_accuracy: 0.9731 - val_loss: 0.0871
Epoch 6/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - accuracy: 0.9988 - loss: 0.0087 - val_accuracy: 0.9753 - val_loss: 0.0880
Epoch 7/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - accuracy: 0.9993 - loss: 0.0064 - val_accuracy: 0.9798 - val_loss: 0.0864
Epoch 8/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 6s 48ms/step - accuracy: 0.9990 - loss: 0.0064 - val_accu

In [44]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {acc:.4f}")

y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int)
print(classification_report(y_test, y_pred,
      target_names=['Ham', 'Spam']))



Test Accuracy: 0.9821
              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       954
        Spam       0.94      0.93      0.94       161

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [45]:
#predict 

def predict(message):
    cleaned = clean_text(message)
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded   = pad_sequences(sequence, maxlen=MAX_LEN, padding='post')
    prob     = model.predict(padded, verbose=0)[0][0]
    label    = "SPAM" if prob > 0.5 else "HAM"
    print(f"{label} ({prob:.2f}) → {message}")


predict("Free entry win prize claim now urgent!")
predict("Hey are you coming for lunch today?")
predict("Congratulations you won 1000 pounds!")
predict("Meeting at office tomorrow at 10am")

SPAM (1.00) → Free entry win prize claim now urgent!
HAM (0.00) → Hey are you coming for lunch today?
HAM (0.01) → Congratulations you won 1000 pounds!
HAM (0.00) → Meeting at office tomorrow at 10am
